In [63]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
import numpy as np
from sklearn.metrics import r2_score

In [3]:
df = pd.read_csv('../data/cleaned_data/vgchartz-2024-cleaned.csv')
df.head()

,img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update,has_critic_score,year
0,/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,2013-09-17,NaN,1,2013.0
1,/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,2014-11-18,2018-01-03,1,2014.0
2,/games/boxart/827563ccc.jpg,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,2002-10-28,NaN,1,2002.0
3,/games/boxart/full_9218923AmericaFrontccc.jpg,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,NaN,15.86,9.06,0.06,5.33,1.42,2013-09-17,NaN,0,2013.0
4,/games/boxart/full_4990510AmericaFrontccc.jpg,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,8.1,15.09,6.18,0.41,6.05,2.44,2015-11-06,2018-01-14,1,2015.0


In [14]:
df.isna().sum()

title                   0
console                 0
genre                   0
publisher               0
developer              17
critic_score        57338
total_sales             0
na_sales                0
jp_sales                0
pal_sales               0
other_sales             0
release_date         7051
last_update         46137
has_critic_score        0
year                 7051
dtype: int64

In [25]:
df.corr()

ValueError: could not convert string to float: 'Grand Theft Auto V'

In [21]:


# df_rock = pd.get_dummies(df, columns=['console', 'genre'], drop_first=True)
# df.drop(columns=['img'], inplace=True)


# df.isna().sum()
imputer = KNNImputer(n_neighbors=5, missing_values=np.nan, weights="uniform", add_indicator=False)
imputer.fit(df['critic_score'].values.reshape(-1, 1))
imputed_vals_critic = imputer.transform(df['critic_score'].values.reshape(-1, 1))


In [22]:
df['critic_score_new'] = imputed_vals_critic
df.isna().sum()

title                   0
console                 0
genre                   0
publisher               0
developer              17
critic_score        57338
total_sales             0
na_sales                0
jp_sales                0
pal_sales               0
other_sales             0
release_date         7051
last_update         46137
has_critic_score        0
year                 7051
critic_score_new        0
dtype: int64

In [35]:
df.dtypes
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
num_cols

Index(['critic_score', 'total_sales', 'na_sales', 'jp_sales', 'pal_sales',
       'other_sales', 'has_critic_score', 'year', 'critic_score_new'],
      dtype='object')

In [33]:
# df[num_cols].corr()
corr_matrix = df.corr(numeric_only=True).abs()

high_corr_pairs = [
    (col1, col2, corr_matrix.loc[col1, col2])
    for col1 in corr_matrix.columns
    for col2 in corr_matrix.columns
    if col1 < col2 and corr_matrix.loc[col1, col2] > 0.7
]
high_corr_pairs


[('critic_score', 'critic_score_new', np.float64(1.0)),
 ('na_sales', 'total_sales', np.float64(0.9220972623428461)),
 ('na_sales', 'pal_sales', np.float64(0.7190193894038731)),
 ('na_sales', 'other_sales', np.float64(0.7167348033026676)),
 ('pal_sales', 'total_sales', np.float64(0.9089522550981662)),
 ('other_sales', 'total_sales', np.float64(0.8660855998494708)),
 ('other_sales', 'pal_sales', np.float64(0.8327998304542309))]

In [42]:
imputer = SimpleImputer(strategy='median', missing_values=np.nan)
imputer.fit(df[['critic_score', 'year']])
fitted_cols = imputer.transform(df[['critic_score', 'year']])

In [45]:
fitted_cols
df[['critic_new_new', 'year_new']] = fitted_cols

In [46]:
df.isna().sum()

title                   0
console                 0
genre                   0
publisher               0
developer              17
critic_score        57338
total_sales             0
na_sales                0
jp_sales                0
pal_sales               0
other_sales             0
release_date         7051
last_update         46137
has_critic_score        0
year                 7051
critic_score_new        0
critic_new_new          0
year_new                0
dtype: int64

In [52]:


df.drop(columns=['critic_score', 'year', 'last_update', 'release_date', 'developer'], inplace=True)

In [53]:
df.isna().sum()

title               0
console             0
genre               0
publisher           0
total_sales         0
na_sales            0
jp_sales            0
pal_sales           0
other_sales         0
has_critic_score    0
critic_score_new    0
critic_new_new      0
year_new            0
dtype: int64

## 2nd attempt at regression with imputing scores

In [54]:
df_dropped = df.drop(columns=['title', 'publisher', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales', 'critic_score_new'])
df_4_regression = pd.get_dummies(df_dropped, columns=['console', 'genre'], drop_first=True)



In [55]:
total_sales = df_4_regression['total_sales']
df_4_regression_wo_label = df_4_regression.drop(columns=['total_sales'])
X_train, X_test, y_train, y_test = train_test_split(df_4_regression_wo_label, total_sales, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((51212, 102), (12804, 102), (51212,), (12804,))

In [56]:
model = LinearRegression().fit(X_train, y_train)

In [59]:
print(model.score(X_train, y_train))
print(model.score(X_test, y_test))
print(len(df_4_regression.columns))


0.1491640558223145
0.1429127972539671
103


## Lets try by log transforming my target variable

In [60]:
y_log = np.log1p(y_train)

model = LinearRegression().fit(X_train, y_log)

In [65]:
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)

r2_score(y_test, y_pred)

0.13081766973258213

### A linear regression model does not really work here
### We have imputed values and removed NAN entries while also one hot encoding. 
### Even amongst these adjustments are model is not predicting total sales very well
### We are going to try some nonlinear models here

In [68]:

df_4_regression.to_csv('../data/cleaned_data/vgchartz-2024-cleaned-imputed.csv', index=False)